# Optuna HPO trên Google Colab — CIES Fraud Detection

Notebook độc lập, chạy song song với `kaggle_optuna_tuning.ipynb`. Dùng khi cần thêm phiên GPU
(Kaggle giới hạn 30 giờ GPU/tuần và 9 giờ/phiên) hoặc muốn tiếp tục 1 model đang tune dở.

**Khác Kaggle ở 2 điểm quan trọng:**
1. Đĩa của Colab (`/content/...`) bị xoá sạch mỗi khi runtime kết thúc — checkpoint và kết quả
   phải lưu vào **Google Drive**, không phải `/content`.
2. Colab miễn phí cần trình duyệt còn kết nối để runtime tiếp tục chạy. Tắt hẳn máy (không phải
   gập nắp còn bật) nhiều khả năng làm runtime bị ngắt. Nhờ checkpoint sau từng trial, không mất
   dữ liệu đã chạy — chỉ cần mở lại notebook và chạy tiếp.

Bật GPU trước khi chạy: **Runtime → Change runtime type → T4 GPU**.

## 1. Gắn Google Drive (lưu checkpoint + kết quả)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/cies-tuning'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/tuning_checkpoints', exist_ok=True)
print('Checkpoint/kết quả sẽ lưu vào:', DRIVE_DIR)

### Tiếp tục checkpoint đã có từ Kaggle (nếu có)

Nếu bạn đã tune dở `ann` trên Kaggle, tải `results/tuning_checkpoints/ann.db` từ tab Output của
phiên Kaggle đó, rồi upload vào `MyDrive/cies-tuning/tuning_checkpoints/ann.db` (qua giao diện Drive,
hoặc ô upload bên dưới). Notebook sẽ tiếp tục đúng số trial đã có — **không chạy trùng lặp với
Kaggle**. Chỉ nên tune 1 model ở 1 nơi tại 1 thời điểm; đừng chạy đồng thời cả Kaggle lẫn Colab
cho cùng model, vì mỗi bên sẽ tự tính n_trials riêng, phá vỡ ngân sách 100 trial thống nhất.

In [ ]:
# Bỏ qua cell này nếu bạn ĐÃ upload ann.db vào Drive qua giao diện web.
# Ngược lại, chạy cell này để upload trực tiếp từ máy.
from google.colab import files
import shutil

uploaded = files.upload()  # chọn ann.db (nếu có)
for name in uploaded:
    shutil.move(name, f'{DRIVE_DIR}/tuning_checkpoints/{name}')
    print(f'Đã đặt {name} vào {DRIVE_DIR}/tuning_checkpoints/')

## 2. Clone code

In [ ]:
import os

REPO_URL = "https://github.com/gnUrt1106/fraud-detection-CIES.git"
REPO_DIR = "/content/fraud-detection-CIES"

if not os.path.isdir(REPO_DIR):
    os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
os.system("git pull -q")
print("Đang ở:", os.getcwd())

In [ ]:
# Colab đã có sẵn hầu hết (pandas, sklearn, xgboost, torch...).
!pip install -q "optuna>=3.4.0" "catboost>=1.2" "pyarrow>=14.0.0"

## 3. Chuẩn bị dữ liệu

Cần `train_encoded.parquet` (sinh ở `notebooks/02_preprocessing.ipynb`, hoặc lấy từ dataset Kaggle
`cies-processed`). Upload file này (khoảng 50MB) vào `MyDrive/cies-tuning/train_encoded.parquet`
một lần — các lần chạy sau không cần upload lại.

In [ ]:
import glob
import shutil

from src.config import PROCESSED_DATA_DIR

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
dst_path = PROCESSED_DATA_DIR / "train_encoded.parquet"
drive_data = f"{DRIVE_DIR}/train_encoded.parquet"

if os.path.exists(drive_data):
    shutil.copy(drive_data, dst_path)
    print(f"Đã copy từ Drive: {drive_data} -> {dst_path}")
elif dst_path.exists():
    print(f"Đã có sẵn: {dst_path}")
else:
    print(
        f"KHÔNG tìm thấy {drive_data}.\n"
        "-> Upload train_encoded.parquet vào MyDrive/cies-tuning/ (qua giao diện Drive), "
        "rồi chạy lại cell này."
    )

## 4. Config — chọn model cần tune

In [ ]:
# Vùng tìm và N_TRIALS PHẢI khớp notebooks/kaggle_optuna_tuning.ipynb để cùng ngân sách/giao
# thức. ANN: epochs đã nới 10-100 (bước 10) sau khi checkpoint Kaggle chạm biên cũ (60) ở trial 15.
MODELS_TO_TUNE = ["ann"]

N_TRIALS = 100
N_SPLITS = 5

# Dừng MỀM sau trial đang chạy khi hết ngân sách — không đổi N_TRIALS nên không ảnh hưởng tính
# công bằng. 4 giờ vì Colab miễn phí cần trình duyệt còn kết nối; chạy lại nhiều lượt để tiếp tục.
SESSION_BUDGET = 4 * 3600

## 5. Load dữ liệu + chạy Optuna HPO (checkpoint vào Drive)

In [ ]:
import pandas as pd

from src.config import PROCESSED_DATA_DIR, TARGET_COL

train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train_encoded.parquet")
X = train_df.drop(columns=[TARGET_COL]).values
y = train_df[TARGET_COL].values

print(f"Train shape: {X.shape}, fraud rate: {y.mean():.4%}")

In [ ]:
from pathlib import Path

from src.models.tune import tune_all_models

all_results = tune_all_models(
    X, y,
    models=MODELS_TO_TUNE,
    n_trials=N_TRIALS,
    n_splits=N_SPLITS,
    output_dir=Path(DRIVE_DIR),                       # best_params.json lưu trên Drive
    checkpoint_dir=Path(DRIVE_DIR) / "tuning_checkpoints",
    session_budget=SESSION_BUDGET,
)

## 6. Kết quả

`best_params.json` nằm ở `MyDrive/cies-tuning/best_params.json`. Model chỉ được ghi vào đó khi ĐỦ
`N_TRIALS` — nếu hết `SESSION_BUDGET` giữa chừng, mở lại notebook này (Runtime mới, chạy lại từ
đầu) để tiếp tục; checkpoint trên Drive không mất.

Khi `ann` đủ 100 trial: tải `best_params.json` từ Drive, đưa vào `results/` của repo local để merge
cùng 4 model kia (giống cách merge kết quả Kaggle trước đó).

In [ ]:
for model_name, res in all_results.items():
    if "best_pr_auc" in res:
        print(f"{model_name}: best PR-AUC = {res['best_pr_auc']:.4f} ({res['n_trials']} trials)")
    else:
        print(f"{model_name}: {res}")